# Day 08：Attention Mask 可视化

对应 [`docs/day07-08.md`](../docs/day07-08.md) 的**任务 6**。

本实验验证三件事：

1. Prefill：`Sq=Skv=4` 时 causal mask 是下三角
2. Decode：`Sq=1, Skv=5` 时最新 Query 可以看到全部历史 Key
3. Padding：PAD Key 被转换成 additive mask 的极小值

## 1. Prefill causal mask

打印 `Sq=Skv=4` 的可见性矩阵：`■` 表示可见，`·` 表示屏蔽。

In [ ]:
import torch

from mini_transformer.attention import build_causal_mask


prefill_mask = build_causal_mask(
    query_len=4,
    key_value_len=4,
    device=torch.device("cpu"),
    dtype=torch.float32,
)
is_visible = prefill_mask == 0

print("additive mask:")
print(prefill_mask)
print("\n可见性矩阵（■ 可见，· 屏蔽）：")
for row in is_visible.tolist():
    print(" ".join("■" if is_allowed else "·" for is_allowed in row))

## 2. Decode causal mask

打印 `Sq=1, Skv=5` 的可见性矩阵，确认唯一一行全部可见。

In [ ]:
import torch

from mini_transformer.attention import build_causal_mask


decode_mask = build_causal_mask(
    query_len=1,
    key_value_len=5,
    device=torch.device("cpu"),
    dtype=torch.float32,
)
is_visible = decode_mask == 0

print("additive mask:")
print(decode_mask)
print("\n可见性矩阵（■ 可见，· 屏蔽）：")
for row in is_visible.tolist():
    print(" ".join("■" if is_allowed else "·" for is_allowed in row))

assert is_visible.all(), "Decode 的最新 Query 应能看到全部历史 Key"

## 3. Padding mask

把两条不同有效长度的 1/0 mask 转成 additive mask，检查 shape 和屏蔽位置。

In [ ]:
import torch

from mini_transformer.attention import build_padding_mask


attention_mask = torch.tensor(
    [
        [1, 1, 1, 0],
        [1, 1, 0, 0],
    ],
    dtype=torch.long,
)
padding_mask = build_padding_mask(attention_mask, dtype=torch.float32)

print("输入 1/0 mask:")
print(attention_mask)
print(f"\n输入 shape: {tuple(attention_mask.shape)}")
print(f"输出 shape: {tuple(padding_mask.shape)}")
print("\nadditive padding mask:")
print(padding_mask)

print("\n每条序列的 Key 可见性（■ 可见，· 屏蔽）：")
for batch_index, row in enumerate((padding_mask[:, 0, 0, :] == 0).tolist()):
    symbols = " ".join("■" if is_allowed else "·" for is_allowed in row)
    print(f"batch {batch_index}: {symbols}")

assert padding_mask.shape == (2, 1, 1, 4)